## 1. Installation et imports

In [39]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 259 kB in 2s (159 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
54 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Ski

In [40]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

run_ollama_serve()

In [41]:
!ollama pull llama3.2

In [42]:
# Installation de la bibliothèque Ollama si nécessaire
!pip install ollama requests

In [43]:
import json
import requests
from typing import Dict, List, Optional
from dataclasses import dataclass

# Vérifier si ollama package est disponible
try:
    import ollama
    OLLAMA_PACKAGE_AVAILABLE = True
    print("✓ Package ollama disponible")
except ImportError:
    OLLAMA_PACKAGE_AVAILABLE = False
    print("⚠ Package ollama non disponible, utilisation de requests")

✓ Package ollama disponible


## 2. Définition du schéma du graphe RDF

In [44]:
@dataclass
class GraphSchema:
    """Représente le schéma de notre graphe RDF"""

    # Vocabulaires et préfixes
    prefixes = """
PREFIX schema: <https://schema.org/>
PREFIX ex: <http://example.org/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    """

    # Description du schéma
    schema_description = """
Le graphe de connaissances contient des informations sur les villes, pays, espaces verts et pollution de l'air.

Classes principales:
- schema:City - Représente une ville
- schema:Country - Représente un pays

Propriétés clés pour les villes:
- schema:name - Nom de la ville (string)
- ex:cityCode - Code identifiant de la ville (string)
- ex:countryName - Nom du pays où se trouve la ville (string)
- ex:SDGRegion - Region dans laquelle se trouve la ville (string)
- ex:greenShare - Pourcentage de couverture d'espaces verts en 2020 (decimal)
- ex:greenPerCapita - Surface d'espaces verts en m² par habitant en 2020 (decimal)
- ex:aqiValue - Valeur de l'indice de qualité de l'air (integer)
- ex:aqiCategory - Catégorie de qualité de l'air (string: Good, Moderate, Unhealthy, etc.)
- ex:coValue - Valeur de monoxyde de carbone (decimal)
- ex:o3Value - Valeur d'ozone (decimal)
- ex:no2Value - Valeur de dioxyde d'azote (decimal)
- ex:pm25Value - Valeur des particules PM2.5 (decimal)
- ex:pm10Value - Valeur des particules PM10 (decimal)

Exemples d'URIs:
- Villes: http://example.org/city/paris
- Pays: http://example.org/country/france
    """

schema = GraphSchema()
print("Schéma du graphe RDF défini")
print(schema.prefixes)

Schéma du graphe RDF défini

PREFIX schema: <https://schema.org/>
PREFIX ex: <http://example.org/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    


## 3. Classe OllamaClient pour communiquer avec Ollama

In [45]:
class OllamaClient:
    """Client pour communiquer avec Ollama en local"""

    def __init__(self, model: str = "llama2", base_url: str = "http://localhost:11434"):
        """
        Initialise le client Ollama.

        Args:
            model: Nom du modèle Ollama à utiliser (llama2, mistral, codellama, etc.)
            base_url: URL de l'API Ollama locale
        """
        self.model = model
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"

    def is_available(self) -> bool:
        """Vérifie si Ollama est disponible"""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=2)
            return response.status_code == 200
        except:
            return False

    def list_models(self) -> List[str]:
        """Liste les modèles disponibles dans Ollama"""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            if response.status_code == 200:
                data = response.json()
                return [model['name'] for model in data.get('models', [])]
            return []
        except:
            return []

    def generate(self, prompt: str, temperature: float = 0.3) -> str:
        """
        Génère une réponse avec Ollama.

        Args:
            prompt: Le prompt à envoyer
            temperature: Température pour la génération (0-1, plus bas = plus déterministe)

        Returns:
            Texte généré par le modèle
        """
        if OLLAMA_PACKAGE_AVAILABLE:
            # Utiliser le package ollama si disponible
            try:
                response = ollama.generate(model=self.model, prompt=prompt)
                return response['response']
            except Exception as e:
                print(f"Erreur avec package ollama: {e}")
                return self._generate_with_requests(prompt, temperature)
        else:
            return self._generate_with_requests(prompt, temperature)

    def _generate_with_requests(self, prompt: str, temperature: float) -> str:
        """Génère avec l'API Ollama via requests"""
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature
            }
        }

        try:
            response = requests.post(self.api_url, json=payload, timeout=60)
            if response.status_code == 200:
                return response.json()['response']
            else:
                return f"Erreur: {response.status_code} - {response.text}"
        except Exception as e:
            return f"Erreur de connexion à Ollama: {str(e)}"

print("Classe OllamaClient définie")

Classe OllamaClient définie


## 4. Vérification de la disponibilité d'Ollama

In [46]:
# Tester la connexion à Ollama
client = OllamaClient(model="llama3.2:latest")  # Vous pouvez changer le modèle ici

if client.is_available():
    print("✓ Ollama est disponible!")
    models = client.list_models()
    print(f"\nModèles disponibles: {models}")
    if models:
        print(f"\nUtilisation du modèle: {client.model}")
    else:
        print("\n⚠ Aucun modèle trouvé. Téléchargez un modèle avec: ollama pull mistral")
else:
    print("✗ Ollama n'est pas disponible")
    print("\nAssurez-vous que:")
    print("1. Ollama est installé (https://ollama.ai/)")
    print("2. Le serveur Ollama est lancé")
    print("3. Un modèle est téléchargé (ex: ollama pull mistral)")

✓ Ollama est disponible!

Modèles disponibles: ['llama3.2:latest']

Utilisation du modèle: llama3.2:latest


## 5. Classe de traduction LLM vers SPARQL

In [47]:
class LLMtoSPARQL:
    """Traduit des questions en langage naturel vers SPARQL en utilisant Ollama"""

    def __init__(self, ollama_client: OllamaClient, use_mock: bool = False):
        """
        Initialise le traducteur.

        Args:
            ollama_client: Instance de OllamaClient
            use_mock: Si True, utilise des réponses mock au lieu d'Ollama
        """
        self.schema = GraphSchema()
        self.ollama = ollama_client
        self.use_mock = use_mock or not ollama_client.is_available()

        if self.use_mock:
            print("⚠ Mode MOCK activé (Ollama non disponible)")

    def create_prompt(self, question: str) -> str:
        """
        Crée un prompt détaillé pour le LLM avec contexte de schéma et exemples.

        Args:
            question: Question en langage naturel

        Returns:
            Prompt formaté pour le LLM
        """
        prompt = f"""Tu es un générateur de requêtes SPARQL. Convertis les questions en langage naturel en requêtes SPARQL valides.

{self.schema.schema_description}

Préfixes SPARQL à utiliser:
{self.schema.prefixes}

Exemples (few-shot learning):

Question: "Quelles villes ont plus de 10% d'espaces verts?"
SPARQL:
{self.schema.prefixes}
SELECT ?city ?cityName ?greenShare
WHERE {{
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:greenShare ?greenShare .
    FILTER(?greenShare > 10)
}}
ORDER BY DESC(?greenShare)

Question: "Quel est l'indice de qualité de l'air moyen dans les villes européennes?"
SPARQL:
{self.schema.prefixes}
SELECT (AVG(?aqi) AS ?avgAQI)
WHERE {{
    ?city rdf:type schema:City ;
          ex:countryName ?country ;
          ex:SDGRegion ?region ;
          ex:aqiValue ?aqi .
    FILTER(?region = "Europe")
}}

Question: "Trouve les villes avec une bonne qualité d'air et beaucoup d'espaces verts par habitant"
SPARQL:
{self.schema.prefixes}
SELECT ?city ?cityName ?aqiCategory ?greenPerCapita
WHERE {{
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:aqiCategory ?aqiCategory ;
          ex:greenPerCapita ?greenPerCapita .
    FILTER(?aqiCategory = "Good" && ?greenPerCapita > 10)
}}
ORDER BY DESC(?greenPerCapita)

Maintenant traduis cette question:
Question: "{question}"

Réponds UNIQUEMENT avec la requête SPARQL, sans explications supplémentaires. Commence directement avec PREFIX.
SPARQL:
"""
        return prompt

    def translate_to_sparql(self, question: str) -> Dict[str, any]:
        """
        Traduit une question en langage naturel vers SPARQL.

        Args:
            question: Question en langage naturel

        Returns:
            Dictionnaire avec la requête SPARQL et métadonnées
        """
        if self.use_mock:
            return self._mock_translation(question)

        # Utiliser Ollama
        prompt = self.create_prompt(question)
        print(f"\n🤖 Envoi de la question à Ollama ({self.ollama.model})...")

        response = self.ollama.generate(prompt, temperature=0.1)

        # Extraire la requête SPARQL de la réponse
        sparql_query = self._extract_sparql(response)

        return {
            "question": question,
            "sparql_query": sparql_query,
            "raw_response": response,
            "method": f"ollama_{self.ollama.model}",
            "confidence": "medium"
        }

    def _extract_sparql(self, response: str) -> str:
        """Extrait la requête SPARQL de la réponse du LLM"""
        # Nettoyer la réponse
        response = response.strip()

        # Si la réponse commence par PREFIX, c'est bon
        if response.startswith("PREFIX"):
            return response

        # Sinon, chercher la partie SPARQL
        lines = response.split('\n')
        sparql_lines = []
        in_sparql = False

        for line in lines:
            if line.strip().startswith("PREFIX") or line.strip().startswith("SELECT"):
                in_sparql = True
            if in_sparql:
                sparql_lines.append(line)

        return '\n'.join(sparql_lines) if sparql_lines else response

    def _mock_translation(self, question: str) -> Dict[str, any]:
        """Traduction mock pour démonstration"""
        question_lower = question.lower()

        if "pollution" in question_lower and ("élevés" in question_lower or "highest" in question_lower):
            query = f"""{self.schema.prefixes}
SELECT ?city ?cityName ?aqiValue ?country
WHERE {{
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:aqiValue ?aqiValue ;
          ex:countryName ?country .
}}
ORDER BY DESC(?aqiValue)
LIMIT 10"""

        elif "vert" in question_lower and ("moins" in question_lower or "low" in question_lower):
            query = f"""{self.schema.prefixes}
SELECT ?city ?cityName ?greenShare ?greenPerCapita
WHERE {{
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:greenShare ?greenShare ;
          ex:greenPerCapita ?greenPerCapita .
    FILTER(?greenShare < 5)
}}
ORDER BY ?greenShare
LIMIT 20"""

        elif "corrélation" in question_lower or "relationship" in question_lower:
            query = f"""{self.schema.prefixes}
SELECT ?cityName ?greenShare ?aqiValue
WHERE {{
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:greenShare ?greenShare ;
          ex:aqiValue ?aqiValue .
}}
ORDER BY ?greenShare"""

        else:
            query = f"""{self.schema.prefixes}
SELECT ?city ?cityName ?country ?greenShare ?aqiValue
WHERE {{
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:countryName ?country .
    OPTIONAL {{ ?city ex:greenShare ?greenShare }}
    OPTIONAL {{ ?city ex:aqiValue ?aqiValue }}
}}
LIMIT 100"""

        return {
            "question": question,
            "sparql_query": query,
            "method": "mock",
            "confidence": "medium"
        }

print("Classe LLMtoSPARQL définie")

Classe LLMtoSPARQL définie


## 6. Questions de compétence

Nous testons 3 questions qui interrogent différents aspects du graphe:

In [48]:
COMPETENCY_QUESTIONS = [
    {
        "id": 1,
        "question": "Quelles sont les 50 villes avec les niveaux de pollution les plus élevés?",
        "aspect": "Air pollution ranking",
        "description": "Teste la capacité à interroger les données de pollution et effectuer un tri"
    },
    {
        "id": 2,
        "question": "Quelles villes ont moins de 5% d'espaces verts en Europe?",
        "aspect": "Green area filtering",
        "description": "Teste la capacité à filtrer sur un seuil numérique"
    },
    {
        "id": 3,
        "question": "Y a-t-il une corrélation entre les espaces verts et la qualité de l'air?",
        "aspect": "Multi-dimensional analysis",
        "description": "Teste la capacité à interroger plusieurs propriétés pour analyse de corrélation"
    }
]

print("Questions de compétence définies:")
for cq in COMPETENCY_QUESTIONS:
    print(f"  {cq['id']}. {cq['question']}")

Questions de compétence définies:
  1. Quelles sont les 50 villes avec les niveaux de pollution les plus élevés?
  2. Quelles villes ont moins de 5% d'espaces verts en Europe?
  3. Y a-t-il une corrélation entre les espaces verts et la qualité de l'air?


## 7. Test des traductions avec Ollama

In [49]:
# Initialiser le traducteur
translator = LLMtoSPARQL(client, use_mock=False)

print("="*80)
print("Test des traductions LLM vers SPARQL")
print("="*80)

Test des traductions LLM vers SPARQL


### Question 1: Ranking de pollution

In [56]:
cq1 = COMPETENCY_QUESTIONS[0]
print(f"\n{'='*80}")
print(f"Question #{cq1['id']}: {cq1['question']}")
print(f"Aspect testé: {cq1['aspect']}")
print(f"{'='*80}\n")

result1 = translator.translate_to_sparql(cq1['question'])

print("\nRequête SPARQL générée:")
print("-"*80)
print(result1['sparql_query'])
print("-"*80)


Question #1: Quelles sont les 50 villes avec les niveaux de pollution les plus élevés?
Aspect testé: Air pollution ranking


🤖 Envoi de la question à Ollama (llama3.2:latest)...

Requête SPARQL générée:
--------------------------------------------------------------------------------
PREFIX schema: <https://schema.org/>
PREFIX ex: <http://example.org/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?city ?cityName ?aqiValue
WHERE {
    ?city rdf:type schema:City ;
          ex:aqiValue ?aqiValue .
    FILTER(COUNT(?city) = 50)
}
ORDER BY DESC(?aqiValue)
--------------------------------------------------------------------------------


### Question 2: Filtrage espaces verts

In [51]:
cq2 = COMPETENCY_QUESTIONS[1]
print(f"\n{'='*80}")
print(f"Question #{cq2['id']}: {cq2['question']}")
print(f"Aspect testé: {cq2['aspect']}")
print(f"{'='*80}\n")

result2 = translator.translate_to_sparql(cq2['question'])

print("\nRequête SPARQL générée:")
print("-"*80)
print(result2['sparql_query'])
print("-"*80)


Question #2: Quelles villes ont moins de 5% d'espaces verts en Europe?
Aspect testé: Green area filtering


🤖 Envoi de la question à Ollama (llama3.2:latest)...

Requête SPARQL générée:
--------------------------------------------------------------------------------
PREFIX schema: <https://schema.org/>
PREFIX ex: <http://example.org/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?city ?cityName ?greenShare
WHERE {
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:greenShare ?greenShare .
    FILTER(?greenShare < 5 AND ex:SDGRegion = "Europe")
}
ORDER BY DESC(?greenShare)
--------------------------------------------------------------------------------


### Question 3: Analyse de corrélation

In [58]:
cq3 = COMPETENCY_QUESTIONS[2]
print(f"\n{'='*80}")
print(f"Question #{cq3['id']}: {cq3['question']}")
print(f"Aspect testé: {cq3['aspect']}")
print(f"{'='*80}\n")

result3 = translator.translate_to_sparql(cq3['question'])

print("\nRequête SPARQL générée:")
print("-"*80)
print(result3['sparql_query'])
print("-"*80)


Question #3: Y a-t-il une corrélation entre les espaces verts et la qualité de l'air?
Aspect testé: Multi-dimensional analysis


🤖 Envoi de la question à Ollama (llama3.2:latest)...

Requête SPARQL générée:
--------------------------------------------------------------------------------
PREFIX schema: <https://schema.org/>
PREFIX ex: <http://example.org/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?greenShare ?aqiValue
WHERE {
    ?city rdf:type schema:City ;
          ex:greenShare ?greenShare ;
          ex:aqiValue ?aqiValue .
    FILTER(?aqiValue > 0)
}

ORDER BY DESC(?aqiValue)
--------------------------------------------------------------------------------


## 8. Exemple de prompt envoyé au LLM

Voyons le prompt complet qui est envoyé à Ollama:

In [53]:
example_prompt = translator.create_prompt("Quelles sont les villes les plus polluées?")
print("="*80)
print("Exemple de prompt envoyé à Ollama")
print("="*80)
print(example_prompt)

Exemple de prompt envoyé à Ollama
Tu es un générateur de requêtes SPARQL. Convertis les questions en langage naturel en requêtes SPARQL valides.


Le graphe de connaissances contient des informations sur les villes, pays, espaces verts et pollution de l'air.

Classes principales:
- schema:City - Représente une ville
- schema:Country - Représente un pays

Propriétés clés pour les villes:
- schema:name - Nom de la ville (string)
- ex:cityCode - Code identifiant de la ville (string)
- ex:countryName - Nom du pays où se trouve la ville (string)
- ex:SDGRegion - Region dans laquelle se trouve la ville (string)
- ex:greenShare - Pourcentage de couverture d'espaces verts en 2020 (decimal)
- ex:greenPerCapita - Surface d'espaces verts en m² par habitant en 2020 (decimal)
- ex:aqiValue - Valeur de l'indice de qualité de l'air (integer)
- ex:aqiCategory - Catégorie de qualité de l'air (string: Good, Moderate, Unhealthy, etc.)
- ex:coValue - Valeur de monoxyde de carbone (decimal)
- ex:o3Value - 

## 9. Test interactif

Testez vos propres questions:

In [64]:
# Décommentez et modifiez pour tester vos questions
custom_question = "Quelles sont les villes avec une pollution de catégorie forte et un taux d'espace verts supérieur à 20% ?"
result = translator.translate_to_sparql(custom_question)
print(result['sparql_query'])


🤖 Envoi de la question à Ollama (llama3.2:latest)...
PREFIX schema: <https://schema.org/>
PREFIX ex: <http://example.org/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?city ?cityName ?aqiCategory ?greenShare
WHERE {
    ?city rdf:type schema:City ;
          schema:name ?cityName ;
          ex:aqiValue ?aqi ;
          ex:greenShare ?greenShare .
    FILTER(?aqi = "Unhealthy" && ?greenShare > 20)
}


## 10. Résumé et discussion

### Ce qui fonctionne ✅

- **Questions structurées**: Le LLM excelle avec des questions mappant vers des patterns SPARQL simples
- **Few-shot learning**: Les exemples guident efficacement la génération
- **Schéma bien défini**: Mapping correct des concepts vers propriétés
- **Ollama local**: Pas besoin d'API cloud, données restent privées

### Limitations ⚠️

1. **Ambiguïté sémantique**: "Pollution" peut référer à AQI, PM2.5, CO, etc.
2. **Limites SPARQL**: Les corrélations nécessitent post-traitement
3. **Valeurs manquantes**: Besoin d'OPTIONAL pour propriétés non obligatoires
4. **Qualité variable**: Le LLM peut générer du SPARQL syntaxiquement correct mais sémantiquement incorrect

### Améliorations possibles 🚀

1. **Validation des requêtes**: Vérifier syntaxe et schéma avant exécution
2. **Boucle de raffinement**: Exécuter, détecter erreurs, corriger avec LLM
3. **Plus d'exemples**: Couvrir plus de patterns (UNION, OPTIONAL, sous-requêtes)
4. **Templates de requêtes**: Pour questions fréquentes
5. **Modèles spécialisés**: Fine-tuner un modèle sur génération SPARQL
6. **Hybrid approach**: Templates pour questions simples, LLM pour complexes